# CUPID

## Test the uncertainty estimation performance

In [ ]:
from sklearn.metrics import roc_auc_score, auc
import pandas as pd
import os

In [ ]:
def display_df(df):
    display(df.head())
    print("Shape:", df.shape)


def load_predictions(fp_predictions_file):
    return pd.read_csv(fp_predictions_file, index_col=0)


def calculate_auc(pred_df, ue_col, loss_col):
    ue = pred_df[ue_col]
    loss = pred_df[loss_col]
    return roc_auc_score(loss, ue)


def get_auc(pred_df):
    df_list = []
    ue_type_list = []
    for ue_type, ue_info_dict in stat_dict.items():
        ue_col = ue_info_dict["ue"]
        zero_one_loss_col = ue_info_dict["01loss"]
        df_list.append({
            "AUC": calculate_auc(pred_df, ue_col, zero_one_loss_col),
        })
        ue_type_list.append(ue_type)
    return pd.DataFrame(df_list, index=ue_type_list)


def evaluate_model(pred_df, save_folder):
    save_folder = save_folder
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    fp_ue_file = os.path.join(save_folder, "uncertainty_estimation_performance.csv")
    ue_perf_df = get_auc(pred_df)
    display(ue_perf_df)
    ue_perf_df.to_csv(fp_ue_file)
    return ue_perf_df


def combine_pred_df(pred_df_list):
    column_list = []
    df_list = []
    for pred_df in pred_df_list:
        new_cols = [col for col in pred_df.columns if col not in column_list]
        df_list.append(pred_df[new_cols].reset_index(drop=True))
        column_list.extend(new_cols)
    return pd.concat(df_list, axis=1)

In [ ]:
stat_dict = {
    "CUPID_Alea":{"ue": "var_usum", "01loss":"wronglist"},
    "CUPID_Epis":{"ue":"softoutres", "01loss":"wronglist"},
}

In [ ]:
folder_old = "" # path of predicted results
folder_now = "" # path of save dir
pred_df = load_predictions(folder_old)
pred_df["wronglist"] = pred_df['wronglist'].apply(lambda x: min(x,1))
display_df(pred_df)
evaluate_model(pred_df,save_folder=folder_now)